### **HPC command lines**

Login:
```linux
ssh s224215@login.hpc.dtu.dk
```

Path for folder on HPC:
```linux
/zhome/76/c/186999/02613_PyHPC
```

Path to current folder on MAC:
```linux
/Users/lucialu/Github/DTU_Workspace_MSc/1_semester/python_HPC
```

Example upload:
```linux
scp week_3/HPC_chache_effects.py s224215@login.hpc.dtu.dk:/zhome/76/c/186999/02613_PyHPC
```

Example download:
```linux
scp s224215@login.hpc.dtu.dk:/zhome/76/c/186999/02613_PyHPC/haversine.out week_4/ 
```

### **Chache effects 1**

In [2]:
import numpy as np
from time import perf_counter

SIZE = 100
mat = np.random.rand(SIZE, SIZE)

start = perf_counter()

double_column = 2 * mat[:, 0]
double_row = 2 * mat[0, :]

end = perf_counter()

print(f"Time taken: {end - start:.6f} seconds")

Time taken: 0.000108 seconds


### **Chache effects 2**

```python
#!/bin/bash
#BSUB -J cache_effects_2 / job name
#BSUB -q hpc / queue name
#BSUB -W 2 / wall time limit in minutes
#BSUB -R "span[hosts=1]" / run a single node
#BSUB -R "select[model==XeonGold6126 || model==XeonGold6142 || model==XeonGold6226R]" / select nodes with specific CPU models
#BSUB -R "rusage[mem=1024]" / memory requirement in MB
#BSUB -n 1 / number of cores
#BSUB -B / send email at the beginning of the job
#BSUB -N / send email at the end of the job
#BSUB -o cache_effects_%J.out / output file with job ID
#BSUB -e cache_effects_%J.err / error file with job ID

echo "CPU model on this node:"
grep "model name" /proc/cpuinfo | head -n 1

echo "Running Python script now!"
python3 HPC_chache_effects.py
```

### **Chache effects 3**

Measure time ranging from $10^1$ to $10^{4.5}$.

In [ ]:
import numpy as np
from time import perf_counter

sizes = np.logspace(1, 4.5, num=20).astype(int)

for SIZE in sizes:
    mat = np.random.rand(SIZE, SIZE)

    start = perf_counter()

    double_column = 2 * mat[:, 0]
    double_row = 2 * mat[0, :]

    end = perf_counter()

    print(f"Size: {SIZE}x{SIZE}, Time taken: {end - start:.6f} seconds")

### **Chache effects 4**

We have to do MFLOP/s so:
$$
\text{MFLOP/s} = \frac{\text{Number of floating point operations}}{\text{Execution time in seconds} \times 10^6}
$$
We do everything in our python script. Moreover, this is python line for converting into kilobytes:
```python
size_kb = (8 * SIZE * SIZE) / 1024
```
Since matrix has $N^2$ elements, each element is 8 bytes (for double precision), we multiply by 8 and divide by 1024 to convert to kilobytes.

In [ ]:
import numpy as np
from time import perf_counter
import matplotlib.pyplot as plt

sizes = np.logspace(1, 4.5, num=20).astype(int)

size_kb_list = []
row_perf = []
col_perf = []

repeats = 1000

for SIZE in sizes:
    mat = np.random.rand(SIZE, SIZE)

    # column timing
    start = perf_counter()
    for _ in range(repeats):
        2 * mat[:, 0]
    end = perf_counter()
    col_time = (end - start) / repeats

    # row timing
    start = perf_counter()
    for _ in range(repeats):
        2 * mat[0, :]
    end = perf_counter()
    row_time = (end - start) / repeats

    # Compute performance
    col_mflops = (SIZE / col_time) * 1e-6
    row_mflops = (SIZE / row_time) * 1e-6

    size_kb = (8 * SIZE * SIZE) / 1024

    size_kb_list.append(size_kb)
    row_perf.append(row_mflops)
    col_perf.append(col_mflops)

plt.figure()
plt.loglog(size_kb_list, row_perf, label="Row")
plt.loglog(size_kb_list, col_perf, label="Column")
plt.xlabel("Matrix size (KB)")
plt.ylabel("Performance (MFLOP/s)")
plt.legend()
plt.tight_layout()
plt.savefig("performance.png")

ratio = np.array(row_perf) / np.array(col_perf)

plt.figure()
plt.loglog(size_kb_list, ratio)
plt.xlabel("Matrix size (KB)")
plt.ylabel("Row / Column Performance Ratio")
plt.tight_layout()
plt.savefig("ratio.png")

Then for chache sum `lscpu`:
```linux
Caches (sum of all):         
  L1d:                       512 KiB (16 instances)
  L1i:                       512 KiB (16 instances)
  L2:                        16 MiB (16 instances)
  L3:                        22 MiB (1 instance)
```

### **Chache effects 5**

In [ ]:
import numpy as np
from time import perf_counter
import matplotlib.pyplot as plt

sizes = np.logspace(2, 8, num=20).astype(int)

size_kb_list = []
perf_list = []

repeats = 100

for SIZE in sizes:
    mat = np.random.rand(1, SIZE)
    start = perf_counter()
    for _ in range(repeats):
        2 * mat[0, :]
    end = perf_counter()

    time_per_op = (end - start) / repeats
    mflops = (SIZE / time_per_op) * 1e-6
    size_kb = (8 * SIZE) / 1024

    size_kb_list.append(size_kb)
    perf_list.append(mflops)

plt.figure()
plt.loglog(size_kb_list, perf_list)
plt.xlabel("Vector size (KB)")
plt.ylabel("Performance (MFLOP/s)")
plt.tight_layout()
plt.savefig("row_vector_performance.png")

### **Chache effects 6**

In [ ]:
import numpy as np
from time import perf_counter
import matplotlib.pyplot as plt

sizes = np.logspace(2, 8, num=20).astype(int)

size_kb_list = []
perf_list = []

repeats = 100

for SIZE in sizes:
    mat = np.random.rand(1, SIZE).astype('float32')
    start = perf_counter()
    for _ in range(repeats):
        2 * mat[0, :]
    end = perf_counter()

    time_per_op = (end - start) / repeats
    mflops = (SIZE / time_per_op) * 1e-6
    size_kb = (8 * SIZE) / 1024

    size_kb_list.append(size_kb)
    perf_list.append(mflops)

plt.figure()
plt.loglog(size_kb_list, perf_list)
plt.xlabel("Vector size (KB)")
plt.ylabel("Performance (MFLOP/s)")
plt.axvline(32, linestyle="--")
plt.axvline(1024, linestyle="--")
plt.axvline(22000, linestyle="--")
plt.tight_layout()
plt.savefig("row_vector_performance.png")

See AUTOLAB.

### **Efficient data storage with Blosc 1**

In [ ]:
import os
import blosc
import numpy as np


def write_numpy(arr, file_name):
    np.save(f"{file_name}.npy", arr)
    os.sync()


def write_blosc(arr, file_name, cname="lz4"):
    b_arr = blosc.pack_array(arr, cname=cname)
    with open(f"{file_name}.bl", "wb") as w:
        w.write(b_arr)
    os.sync()


def read_numpy(file_name):
    return np.load(f"{file_name}.npy")


def read_blosc(file_name):
    with open(f"{file_name}.bl", "rb") as r:
        b_arr = r.read()
    return blosc.unpack_array(b_arr)

We write the rest in a python script.

### **Efficient data storage with Blosc 2**

### **Efficient data storage with Blosc 3**

### **Efficient data storage with Blosc 4**

### **Efficient data storage with Blosc 5**